# A-Vision (AvitoTech) в Google Colab

[Модель на HF](https://huggingface.co/AvitoTech/avision) — VLM: фото + текстовый промпт.

**GPU:** *Среда выполнения → T4 GPU*.


In [ ]:
!nvidia-smi 2>/dev/null

Sat Apr  4 12:27:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# device_map="auto" - сработает благодаря accelerate - для автопереноса на GPU
# TODO: посмотри токенизатор русский - на гитхабе - как его отчитать

In [ ]:
# Версии как в requirements-colab.txt (репозиторий 2_photo_scrapper). Без -U.
%pip install -q "transformers>=4.30.0" "accelerate>=1.1.0" "safetensors>=0.4.3" "huggingface-hub>=0.24.0" qwen-vl-utils "bitsandbytes>=0.43.0" "Pillow>=10.2.0,<12" "sentencepiece>=0.2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 22.6 MB/s eta 0:00:00


## Картинка
Вариант A — загрузить файл с компьютера. Вариант B — скачать по URL

In [ ]:
from pathlib import Path

# True — выбрать файл с компьютера; False — скачать по URL ниже
USE_UPLOAD = True

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    assert uploaded, "Выбери файл изображения"
    IMAGE_PATH = Path(list(uploaded.keys())[0])
else:
    import urllib.request
    IMAGE_PATH = Path("/content/sample.jpg")
    _url = "https://images.unsplash.com/photo-1521572163474-6864f9cf17ab?w=800"
    urllib.request.urlretrieve(_url, IMAGE_PATH)

print("Изображение:", IMAGE_PATH.resolve())

Saving пример_фото.jpg to пример_фото.jpg
Изображение: /content/пример_фото.jpg


## Промпт и параметры генерации

In [ ]:
MODEL_ID = "AvitoTech/avision"
PROMPT = "Выпиши какая одежда на человеке на фото."
MAX_NEW_TOKENS = 256

USE_4BIT = True

MIN_PIXELS = 4 * 28 * 28
MAX_PIXELS = 768 * 28 * 28

## Загрузка модели и ответ

In [ ]:
import gc
import os
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

assert torch.cuda.is_available(), "Нужна GPU в Colab"

torch.cuda.empty_cache()
gc.collect()

from google.colab import userdata
_t = userdata.get("HF_TOKEN")
if _t:
  os.environ["HF_TOKEN"] = _t

_hf_kw = {}
if os.environ.get("HF_TOKEN"):
    _hf_kw["token"] = os.environ["HF_TOKEN"]

print(f"Загрузка {MODEL_ID!r} (4-bit={USE_4BIT})…")
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16, # умножения/аккумуляция в bf16 - чтобы уж совсем не терять точность при умножении или сумме
        bnb_4bit_quant_type="nf4", #  тип 4-bit квантизации
        bnb_4bit_use_double_quant=True, #  дополнительное сжатие констант квантизации
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        **_hf_kw,
    )
else:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype="auto",
        device_map="auto",
        **_hf_kw,
    )
processor = AutoProcessor.from_pretrained(MODEL_ID, **_hf_kw)


Загрузка 'AvitoTech/avision' (4-bit=True)…


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

In [ ]:
import re

def clean_avision_output(text: str) -> str:
    """SentencePiece/чат: убрать ▁, заменить <0x0A> на перенос, сжать пробелы."""
    if not text:
        return text
    t = text.replace("<0x0A>", "\n").replace("<0x0a>", "\n")
    t = t.replace("▁", " ")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def avision_ask(prompt, image, max_new_tokens=None):
    if max_new_tokens is None:
        max_new_tokens = MAX_NEW_TOKENS
    if isinstance(image, (str, Path)):
        pil = Image.open(image).convert("RGB")
    else:
        pil = image.convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": pil,
                    "min_pixels": MIN_PIXELS,
                    "max_pixels": MAX_PIXELS,
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]

    chat_text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True #  по этим сообщениям собирается одна большая текстовая строка в формате которую ожидает моделька
    )

    image_inputs, video_inputs = process_vision_info(messages) # вытаскиваются тензоры для картинок в том виде как их ждет Qwen-VL процесср

    inputs = processor( # тут в процессор в батче подаем разом и токены и картинку
        text=[chat_text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens) # полная последовательность токенов сгенеренных моделью (промпт + ответ)

    in_len = inputs["input_ids"].shape[1] # сколько токенов в промпте

    new_tokens = generated_ids[0, in_len:] # ток новые токены (ответ), отрезали промпт, 0 - это индекс батча (единственного)

    tok = getattr(processor, "tokenizer", None) or processor #  объект с методом decode - иногда нужно достать явно (окей..)

    raw = tok.decode( # уже что-то человекочитаемое
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )
    return clean_avision_output(raw) # подчищаем и делаем еще более человекочитаемо

In [ ]:

# PROMPT_2 = "What clothing items do you see in the image? Reply with ONLY a list, nothing else."
PROMPT_2 = "What items of clothing do you see in the image? ANSWER only about clothing."

from google.colab import files
uploaded = files.upload()
assert uploaded, "Выбери файл изображения"
IMAGE_PATH = Path(list(uploaded.keys())[0])

print(avision_ask(PROMPT_2, Image.open(IMAGE_PATH)))

Saving пример_фото_2.jpg to пример_фото_2 (3).jpg
Here's a brief list of the clothing items visible in the image:
- Brown leather jacket
- Black t-shirt
- Dark brown pants
- White and black shoes
- Black belt

These are the main clothing items that can be clearly identified in the image.


In [ ]:
PROMPT_2 = "What items of clothing do you see in the image? ANSWER only about clothing. If you can recognize the brand of closhing - say it"

print(avision_ask(PROMPT_2, Image.open(IMAGE_PATH)))

Here's a list of clothing items visible in the image, with brand recognition for shoes:

1. Brown leather jacket
2. Black t-shirt
3. Brown pants
4. White and black Adidas sneakers

The Adidas sneakers are clearly visible as white and black shoes on the man's feet.


In [ ]:

PROMPT_3 = "Опиши элементы одежды, которые ты видишь на фото. Ничего, кроме одежды, не описывай"

print(avision_ask(PROMPT_3, Image.open(IMAGE_PATH)))

На изображении видно мужчину в коричневой кожаной куртке и брюках, стоящего на улице рядом с магазином. На стекле витрины магазина видна надпись "COFFEE Dessert TOY Alcohol". Мужчина держит телефон, чтобы сделать селфи. На заднем плане можно заметить других людей и здания.


In [ ]:

PROMPT_4 = "Опиши ТОЛЬКО ОДЕЖДУ на фото. Если можешь различить бренд одежды - назови его и конкретную модель одежды"

print(avision_ask(PROMPT_4, Image.open(IMAGE_PATH)))

На изображении видно, что человек стоит на улице перед зданием с витриной магазина. На стекле витрины написано: "COFFEE Dessert TOY Alcohol". Это указывает на то, что в магазине продаются кофе, десерты, игрушки и алкоголь.

Ответ: Магазин продаёт кофе, десерты, игрушки и алкоголь.


In [ ]:
PROMPT_5 = "Назови ТОЛЬКО элементы ОДЕЖДЫ"
print(avision_ask(PROMPT_5, Image.open(IMAGE_PATH)))

На изображении видно, что человек одет в коричневую кожаную куртку и брюки, а также в белую обувь с черными полосками. На заднем плане виден магазин с вывеской "COFFEE Dessert TOY Alcohol". Также на улице можно заметить несколько автомобилей и людей.
